In [0]:

# ============================================================
# DISPLAY GENERATED PYSPARK CODE FOR ALL MAPPINGS
# ============================================================

GENERATED_DIR = (
    "/Volumes/idmc_poc/bronze/idmc_files/generated_pyspark"
)

MAPPINGS = [
    "m_customer_standardize",
    "m_order_enrichment",
    "m_inventory_reconciliation",
    "m_daily_sales_summary"
]

for mapping_name in MAPPINGS:

    file_path = f"{GENERATED_DIR}/{mapping_name}.py"

    print("\n" + "=" * 100)
    print(f"GENERATED PYSPARK: {mapping_name}")
    print("=" * 100)

    try:
        code = dbutils.fs.head(file_path, 1000000)

        print(code)

        print("\nFile:", file_path)
        print("Status: Successfully read")

    except Exception as e:
        print("Unable to read generated code:", str(e))


In [0]:
parsed_path = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "parsed_metadata/idmc_parsed_objects.json"
)

parsed_df = spark.read.option("multiline", "true").json(parsed_path)

display(parsed_df)

In [0]:
# ============================================================
# FINAL POST-MIGRATION VALIDATION - ALL FOUR TARGETS
# ============================================================

from pyspark.sql import functions as F

TARGETS = {
    "m_customer_standardize": {
        "table": "idmc_poc.migration_test.mig4_customer_standardized",
        "keys": ["customer_id"]
    },
    "m_order_enrichment": {
        "table": "idmc_poc.migration_test.mig4_order_enriched",
        "keys": ["order_id"]
    },
    "m_inventory_reconciliation": {
        "table": "idmc_poc.migration_test.mig4_inventory_position",
        "keys": ["product_id", "warehouse_id"]
    },
    "m_daily_sales_summary": {
        "table": "idmc_poc.migration_test.mig4_daily_sales_summary",
        "keys": ["order_date"]
    }
}

final_results = []

for mapping, config in TARGETS.items():

    table = config["table"]
    keys = config["keys"]

    df = spark.table(table)

    null_condition = F.lit(False)

    for key in keys:
        null_condition = null_condition | F.col(key).isNull()

    null_rows = df.filter(null_condition).count()

    duplicate_groups = (
        df.groupBy(*keys)
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    result = {
        "mapping": mapping,
        "target": table,
        "row_count": df.count(),
        "column_count": len(df.columns),
        "null_key_rows": null_rows,
        "duplicate_key_groups": duplicate_groups,
        "status": (
            "PASS"
            if null_rows == 0 and duplicate_groups == 0
            else "REVIEW_REQUIRED"
        )
    }

    final_results.append(result)

    print(result)

print("\nFINAL VALIDATION:",
      "ALL PASS" if all(
          x["status"] == "PASS" for x in final_results
      ) else "REVIEW REQUIRED")

In [0]:
from pathlib import Path

OUTPUT_DIR = Path(
    "/Volumes/idmc_poc/bronze/idmc_files/generated_pyspark"
)

mapping_files = [
    "m_customer_standardize.py",
    "m_order_enrichment.py",
    "m_inventory_reconciliation.py",
    "m_daily_sales_summary.py"
]

for filename in mapping_files:
    file_path = OUTPUT_DIR / filename

    print("\n" + "=" * 100)
    print(f"MAPPING FILE: {filename}")
    print(f"LOCATION: {file_path}")
    print("=" * 100)

    if file_path.exists():
        print(file_path.read_text(encoding="utf-8"))
    else:
        print("File not found:", file_path)